# Meeting 2: The Lanczos Method

## Nandor Kovacs

### 5.3.2026

# Recall: Maximum Eigenvalue Estimation

> **Computational Problem**: Let $A \in \mathbb{H}(\mathbb{R})$ be a real psd matrix. Compute $\lambda_{\max}(A)$

> **Computational Primitive**: We can form $u \rightarrow Au$ for any vector $u \in \mathbb{R}^n$

## The Power Method
* **Input:** Input matrix $A \in \mathbb{M}_n(\mathbb{R})$, initial non-zero vector $b_0 \in \mathbb{R}^n$, number $k$ of iterations
* **Output:** Largest eigenvector estimate $b_k$, largest eigenvalue estimate $\lambda_k$

1. **function** PowerMethod($A, b_0, k$)
2. &emsp;&emsp; $b_0 = \frac{b_0}{\|b_0\|}$
3. &emsp;&emsp; **for** $i = 1, \dots, k$ **do**
4. &emsp;&emsp;&emsp;&emsp; $y_i = A b_{i-1}$
5. &emsp;&emsp;&emsp;&emsp; $b_i = \frac{y_i}{\|y_i\|}$
6. &emsp;&emsp; $\lambda_k = b_k^T A b_k$

***
### Pitfalls: 
- There is a burn in period: for around $\log(n) \cdot \frac{\lambda_1}{\lambda_1 - \lambda_2}$ steps, not much improvement is beeing made.
- If there is no spectral gap, the algorithm is pretty slow.

## Power method in 3D: spectral gap experiments

We now visualize the power method on simple \(3 \times 3\) symmetric matrices, once with a **large spectral gap** and once with a **small spectral gap**. The figures are interactive: rotate them and inspect how the iterates line up with the leading eigenvector.

In [1]:
import numpy as np
import plotly.graph_objects as go


def power_iteration(A: np.ndarray, b0: np.ndarray, num_iters: int) -> np.ndarray:
    """Return the normalized power method iterates b_0, ..., b_num_iters in R^3."""
    b = b0 / np.linalg.norm(b0)
    iters = [b.copy()]
    for _ in range(num_iters):
        b = A @ b
        b = b / np.linalg.norm(b)
        iters.append(b.copy())
    return np.stack(iters, axis=0)


def make_power_figure(iters: np.ndarray,
                      eigenvectors: np.ndarray,
                      title: str,
                      plane_vectors: tuple[np.ndarray, np.ndarray] | None = None) -> go.Figure:
    """Interactive 3D visualization of power method iterates in R^3.

    - "iters" has shape (T, 3) and contains the normalized vectors.
    - "eigenvectors" is a 3x3 orthogonal matrix; first column is the leading eigenvector.
    - If "plane_vectors" is given, shade the plane they span translucently.
    """
    xs, ys, zs = iters[:, 0], iters[:, 1], iters[:, 2]

    fig = go.Figure()

    # Translucent trajectory of all intermediate vectors
    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="lines+markers",
            marker=dict(
                size=4,
                color=np.linspace(0.0, 1.0, len(xs)),
                colorscale="Viridis",
                opacity=0.4,
            ),
            line=dict(color="rgba(50, 50, 200, 0.3)", width=4),
            name="power iterates",
        )
    )

    # Leading eigenvector (opaque)
    v1 = eigenvectors[:, 0]
    fig.add_trace(
        go.Scatter3d(
            x=[0.0, float(v1[0])],
            y=[0.0, float(v1[1])],
            z=[0.0, float(v1[2])],
            mode="lines+markers",
            line=dict(color="red", width=6),
            marker=dict(size=6, color="red"),
            name="leading eigenvector",
        )
    )

    # Last iterate (opaque, slightly different color)
    v_last = iters[-1]
    fig.add_trace(
        go.Scatter3d(
            x=[0.0, float(v_last[0])],
            y=[0.0, float(v_last[1])],
            z=[0.0, float(v_last[2])],
            mode="lines+markers",
            line=dict(color="orange", width=4),
            marker=dict(size=5, color="orange"),
            name="final iterate",
        )
    )

    # Optionally: shade the plane spanned by two eigenvectors
    if plane_vectors is not None:
        u, v = plane_vectors
        grid = np.linspace(-1.0, 1.0, 7)
        U, V = np.meshgrid(grid, grid)
        X = u[0] * U + v[0] * V
        Y = u[1] * U + v[1] * V
        Z = u[2] * U + v[2] * V

        fig.add_trace(
            go.Surface(
                x=X,
                y=Y,
                z=Z,
                opacity=0.3,
                showscale=False,
                colorscale=[[0.0, "rgba(150, 150, 255, 0.3)"], [1.0, "rgba(150, 150, 255, 0.3)"]],
                name="plane of top 2 eigenvectors",
            )
        )

    # Make the axes comparable and the origin visible
    axis_range = dict(showgrid=True, zeroline=True)
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode="data",
            xaxis=axis_range,
            yaxis=axis_range,
            zaxis=axis_range,
        ),
        legend=dict(x=0.02, y=0.98),
    )

    return fig


# Fix a random orthogonal basis (shared between experiments)
np.random.seed(0)
Q, _ = np.linalg.qr(np.random.randn(3, 3))

# Random starting vector (same for both experiments)
b0 = np.random.randn(3)

# --- Large spectral gap experiment ---
# Eigenvalues: lambda1 >> lambda2, lambda3
lambdas_big = np.array([3.0, 1.0, 0.2])
A_big = Q @ np.diag(lambdas_big) @ Q.T

iters_big = power_iteration(A_big, b0, num_iters=15)
fig_big = make_power_figure(
    iters_big,
    eigenvectors=Q,
    title="Power iteration in R^3: large spectral gap",
)
fig_big.show()

# --- Small spectral gap experiment ---
# lambda1 and lambda2 are very close; plane span(e1, e2) is shaded.
lambdas_small = np.array([1.0, 0.98, 0.3])
A_small = Q @ np.diag(lambdas_small) @ Q.T

iters_small = power_iteration(A_small, b0, num_iters=80)
plane_uv = (Q[:, 0], Q[:, 1])
fig_small = make_power_figure(
    iters_small,
    eigenvectors=Q,
    title="Power iteration in R^3: small spectral gap (plane of top 2 eigenvectors shaded)",
    plane_vectors=plane_uv,
)
fig_small.show()

# Mostly Recall: The Krylov Method
**Idea:** Instead of only utilizing $A^k u$ for the eigenvector approximation, we try to make use of the information in the $k-1$ intermediary steps.

> **Def. Krylov subspace**: We call $K_{k+1}(A, \omega) := \text{span}\{\omega, A\omega, A^2 \omega, ..., A^k \omega\}$ the k'th Krylov subspace of $A$ and $\omega$. 

We will usually only write $K_{k+1}$, as it is usually clear which $A$ and $\omega$ we mean.

**Note:** $ u \in K_{k+1} \iff \exists \text{ polynomial of degree at most k } \phi \text{, so that } u = \phi(A)\omega$

## The Randomized Krylov Method (RKM)
We take $\omega$ randomly, and estimate the maximum eigenvalue by the rayleigh quotient
$$\max{\frac{u^*Au}{||u||^2}} = \max{\frac{\omega^*A\phi^2(A)\omega}{\omega^*\phi^2(A)\omega}}$$
Where the equality is evident form $u = \phi(A)\omega$ and $A$ commuting with any polynomial of itself.

This seems like a reasonable thing to do; Previously we estimated our eigenvector in $\langle A^k \omega\rangle \subseteq K_{k+1}$. We hope that by considering the whole space $K_{k+1}$, we get a better result, as hopefully the intermediary steps contain some additional information.

## Bounds
Without a proof, we state the following bounds. Let $A \in \mathbb{H}_n(\mathbb{R})$ and $\xi_k$ be the estimated maximal eigenvalue, then:

- **with a spectral gap**: $\mathbb{E}\text{err}(\xi_k) \leq 2.589\sqrt{n} \left(1 - 2\sqrt{1 - \frac{\lambda_2}{\lambda_1}}\right)^k \leq 2.589\sqrt{n} e^{-2k\sqrt{\gamma}}$

- **without a spectral gap**: $\mathbb{E}\text{err}(\xi_k) \leq 2.575 \left\lceil \frac{\log n}{k} \right\rceil^2$

As a reminder, this is an improvement, in contrast to the RPM bounds:
- **with a spectral gap**: $\mathbb{E}\,\mathrm{err}(\xi_k) \le \sqrt{\frac{(n-1)\pi}{2}} \left(\frac{\lambda_2}{\lambda_1}\right)^k$
- **without a spectral gap**: $\mathbb{E}\,\mathrm{err}(\xi_k)
\le \frac{1}{k} \left[ 1 + \log \sqrt{\frac{(n-1)\pi}{2}} + \log k \right]$

Now, we only pay for the square root of $\varepsilon$ without a spectral gap!

# Maximizing the Rayleigh quotient
## The Krylov scheme
1. Compute an orthonormal basis $Q_k$ for the Kylov subspace $K_k$
2. Compress $A$ to the Krylov subspace: $H_k = Q^*_kAQ_k$
3. Compute the maximal eigenvalue of the compression, $\lambda_{max}(H_k)$

# Arnold iteration
The Arnold iteration is an algorithm for computing $Q_k$. Although it has its own name, it is quite precisely just the gram-schmidt algorithm, orthogonalizing $\{\omega, A\omega, ..., A^k\omega\}$

1. **function** Arnold ($A, \omega$)
2. &emsp;&emsp; $q_1 = \frac{\omega}{||\omega||}$
3. &emsp;&emsp; **for** $i = 2, \dots, k$ **do**
4. &emsp;&emsp;&emsp;&emsp; $\hat{q}_i = Aq_{i-1} - \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k}$
5. &emsp;&emsp;&emsp;&emsp; $q_i = \frac{\hat{q_i}}{||\hat{q_i}||}$

## The algorithm in matrix form
Let $\hat{H}_k$ be the $k+1 \times k$, such that its entry at $i, j$ is equal to the coeffizients arising from the gram-schmidt algorithm in line 4 of the algorithm.

We can easily verify, that $AQ_k = Q_{k+1} \hat{H}_k$.

We note, that $\hat{H}_k$ has only entries on or above the first subdiagonal (upper Hessenberg).
<!-- $$
\frac{\hat{q_i}}{||\hat{q_i}||} = q_i \\
Aq_{i-1} - \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k} = q_i \cdot ||\hat{q_i}||\\
Aq_{i-1} = q_i \cdot ||\hat{q_i}|| + \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k}\\
\\
Aq_i = q_{i+1} \cdot ||\hat{q_{i+1}}|| + \sum_{k=1}^{i}{\langle Aq_{i}, q_k\rangle q_k}
$$ -->



# Lanczos iteration

Now we utilize our assumption from the Computational Problem earlier, that $A$ is self-adjoint, and take a look at the matrix form of the Arnold iteration. Let $H_k$ be the matrix, where we drop the bottom row of $\hat{H_k}$.
$$
\begin{align*}
AQ_k &= Q_{k+1} \hat{H}_k\\
Q_k^*AQ_k &= Q_k^*Q_{k+1} \hat{H}_k\\ 
&= [I_k | Q_k^* q_{k+1}] \hat{H}_k\\
&= [I_k | 0_v] \hat{H}_k \\
&= H_k\\
\end{align*}
$$

Therefore, we note:
- $AQ_k = Q_k H_k$
- $A$ is self adjoint $\implies H_k$ is self adjoint
- $H_k$ self adjoint, and upper Hessenberg $\implies H_k$ symmetric and tridiagonal; $H_k$ is a Jacobi matrix. 

## Algorithm speedup

This gives us a very important insight; During the algorithm, the only non zero inner products $\langle Aq_i, q_k \rangle$ are with $q_i$ and $q_{i-1}$;

$\implies Aq_{i} - \sum_{k=1}^{i}{\langle Aq_{i}, q_k\rangle q_k} = Aq_{i} - \langle Aq_{i}, q_i\rangle q_i - \langle Aq_{i}, q_{i-1}\rangle q_{i-1}$

This results in a speedup by a factor of $O(n)$.

### Note: Implementation detail
Thinking back to the Krylov scheme; We now have a way of computing $Q_k$. Note however, that as a byproduct we also get the second step for "free"; $H_k = Q_k^*AQ_k$ is the compression of $A$ to the Krylov subspace, which we recover during the gram-schmidt orthogonalization.

Recovering $\lambda_{max}(H_k)$ can be done in $O(n)$ steps via the bisection method.

# Demo

In [4]:
import time
import numpy as np


def generate_low_gap_matrix(n: int, gap: float = 1e-3, seed: int | None = 0) -> tuple[np.ndarray, np.ndarray]:
    """Return a symmetric matrix A with a very small spectral gap.

    The eigenvalues are in [0.1, 1], and we enforce
    lambda_1 = 1 and lambda_2 = 1 - gap, with all others smaller.
    The matrix is constructed as A = Q diag(lambdas) Q^T with random orthogonal Q.
    """
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng.standard_normal((n, n)))

    # Start with a smooth spectrum in [0.1, 0.9]
    lambdas = np.linspace(0.1, 0.9, n)
    # Enforce a tiny gap between the two largest eigenvalues
    lambdas[-1] = 1.0
    lambdas[-2] = 1.0 - gap
    lambdas = np.sort(lambdas)  # ascending, largest at the end

    A = Q @ (lambdas * Q.T)
    return A, lambdas


def power_iteration_until_tol(
    A: np.ndarray,
    b0: np.ndarray,
    lam_true: float,
    tol: float = 1e-6,
    max_iters: int = 50_000,
) -> tuple[float, np.ndarray, int, np.ndarray]:
    """Run the power method until the Rayleigh quotient reaches the desired tolerance.

    Returns (lambda_hat, v_hat, num_iters, error_history).
    """
    b = b0 / np.linalg.norm(b0)
    errors: list[float] = []

    for k in range(1, max_iters + 1):
        Ab = A @ b
        b = Ab / np.linalg.norm(Ab)
        lam_hat = float(b @ (A @ b))
        rel_err = abs(lam_hat - lam_true) / abs(lam_true)
        errors.append(rel_err)
        if rel_err < tol:
            return lam_hat, b, k, np.array(errors)

    return lam_hat, b, max_iters, np.array(errors)


def lanczos_until_tol(
    A: np.ndarray,
    b0: np.ndarray,
    lam_true: float,
    tol: float = 1e-6,
    k_max: int = 500,
) -> tuple[float, int, np.ndarray]:
    """Randomized Krylov / Lanczos for the largest eigenvalue.

    We run k steps of Lanczos with starting vector b0, build the tridiagonal
    compression H_k at each step, and stop when the largest eigenvalue of H_k
    reaches the desired relative accuracy.

    Returns (lambda_hat, k, error_history).
    """
    n = A.shape[0]
    v_prev = np.zeros(n)
    v = b0 / np.linalg.norm(b0)

    alphas: list[float] = []
    betas: list[float] = []
    errors: list[float] = []

    for k in range(1, k_max + 1):
        w = A @ v
        alpha = float(v @ w)
        w = w - alpha * v - (betas[-1] * v_prev if k > 1 else 0.0)
        beta = float(np.linalg.norm(w))

        alphas.append(alpha)
        if k < k_max:
            betas.append(beta)

        # Build the current k x k tridiagonal matrix H_k
        H_k = np.diag(alphas)
        if k > 1:
            off = np.array(betas[: k - 1])
            H_k[np.arange(k - 1), np.arange(1, k)] = off
            H_k[np.arange(1, k), np.arange(k - 1)] = off

        # Largest eigenvalue of H_k
        lam_hat = float(np.linalg.eigvalsh(H_k)[-1])
        rel_err = abs(lam_hat - lam_true) / abs(lam_true)
        errors.append(rel_err)

        if rel_err < tol or beta == 0.0 or k == k_max:
            return lam_hat, k, np.array(errors)

        v_prev, v = v, w / beta

    return lam_hat, k_max, np.array(errors)


# --- Experiment setup ---

n = 800          # matrix dimension (tweak as desired)
gap = 1e-3       # spectral gap lambda_1 - lambda_2
tol = 1e-6      # target relative error in lambda_max
seed = 123

A, lambdas = generate_low_gap_matrix(n, gap=gap, seed=seed)
lam_true = float(lambdas[-1])  # ground-truth largest eigenvalue from construction

rng = np.random.default_rng(seed + 1)
b0 = rng.standard_normal(n)

print(f"Matrix size n = {n}")
print(f"Spectral gap (lambda_1 - lambda_2) ≈ {gap}")
print(f"Target relative error: {tol:.1e}\n")

# Power iteration (RPM)
start = time.perf_counter()
lam_power, v_power, iters_power, errs_power = power_iteration_until_tol(
    A, b0, lam_true, tol=tol, max_iters=100_000
)
elapsed_power = time.perf_counter() - start

# Randomized Krylov / Lanczos (RKM)
start = time.perf_counter()
lam_krylov, k_krylov, errs_krylov = lanczos_until_tol(
    A, b0, lam_true, tol=tol, k_max=1_000
)
elapsed_krylov = time.perf_counter() - start

print("Power iteration (RPM):")
print(f"  iterations          = {iters_power}")
print(f"  time                = {elapsed_power:.3f} s")
print(f"  final relative error = {errs_power[-1]:.2e}")

print("\nRandomized Krylov / Lanczos (RKM):")
print(f"  Lanczos steps       = {k_krylov}")
print(f"  time                = {elapsed_krylov:.3f} s")
print(f"  final relative error = {errs_krylov[-1]:.2e}")

Matrix size n = 800
Spectral gap (lambda_1 - lambda_2) ≈ 0.001
Target relative error: 1.0e-06

Power iteration (RPM):
  iterations          = 2144
  time                = 0.091 s
  final relative error = 9.98e-07

Randomized Krylov / Lanczos (RKM):
  Lanczos steps       = 23
  time                = 0.002 s
  final relative error = 4.59e-07


In [6]:
import matplotlib.pyplot as plt  # still used elsewhere
import plotly.graph_objects as go

# Parameter grid (15 x 15)
sizes = np.linspace(200, 2000, 15, dtype=int)
gaps = np.logspace(-1, -3, 15)  # from 1e-1 to 1e-3

tol = 1e-6
base_seed = 123

results = []

for n in sizes:
    for gap in gaps:
        # Build matrix with given size and spectral gap
        A, lambdas = generate_low_gap_matrix(n, gap=gap, seed=base_seed)
        lam_true = float(lambdas[-1])

        # Fresh random start vector for each configuration
        rng = np.random.default_rng(base_seed + n + int(gap * 1e5))
        b0 = rng.standard_normal(n)

        # Power iteration
        start = time.perf_counter()
        lam_power, v_power, iters_power, errs_power = power_iteration_until_tol(
            A, b0, lam_true, tol=tol, max_iters=100_000
        )
        elapsed_power = time.perf_counter() - start

        # Lanczos / randomized Krylov
        start = time.perf_counter()
        lam_krylov, k_krylov, errs_krylov = lanczos_until_tol(
            A, b0, lam_true, tol=tol, k_max=1_000
        )
        elapsed_krylov = time.perf_counter() - start

        results.append(
            {
                "n": n,
                "gap": gap,
                "time_power": elapsed_power,
                "time_lanczos": elapsed_krylov,
            }
        )


In [7]:
# ---- Plotly 3D bar-like stems: power time - Lanczos time ----

gap_values = sorted({r["gap"] for r in results})
size_values = sorted({r["n"] for r in results})

gap_to_idx = {g: j for j, g in enumerate(gap_values)}
size_to_idx = {n: i for i, n in enumerate(size_values)}

x_line, y_line, z_line = [], [], []

for r in results:
    xi = gap_to_idx[r["gap"]]
    yi = size_to_idx[r["n"]]
    hi = r["time_power"] - r["time_lanczos"]

    # vertical segment from z=0 to z=hi, then None to break
    x_line += [xi, xi, None]
    y_line += [yi, yi, None]
    z_line += [0.0, hi, None]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x_line,
            y=y_line,
            z=z_line,
            mode="lines",
            line=dict(color="royalblue", width=6),
        )
    ]
)

fig.update_layout(
    title="Power iteration time minus Lanczos time<br>across sizes and spectral gaps",
    scene=dict(
        xaxis=dict(
            title="Spectral gap",
            tickvals=list(range(len(gap_values))),
            ticktext=[f"{g:.0e}" for g in gap_values],
        ),
        yaxis=dict(
            title="Matrix size n",
            tickvals=list(range(len(size_values))),
            ticktext=[str(n) for n in size_values],
        ),
        zaxis=dict(
            title="Time difference (power - Lanczos) [s]",
        ),
    ),
    width=900,
    height=600,
)

fig.show()

# Spectral functions, and why they're fun
## Spectral function
Let $A \in \mathbb{H}_n$ and $f : I \rightarrow \mathbb{R}, [\lambda_{max}(A), \lambda_{min}(A)] \subseteq I$

Then, the we can define a spectral function as:
$$
f(A) = Uf(\Sigma) V^\top = U\begin{bmatrix}
f(\lambda_1)  &              &        &                  & 0 \\
              & f(\lambda_2) &        &                  &   \\
              &              & \ddots &            &   \\
              &              &  & f(\lambda_{k-1}) & \\
0             &              &        &                  & f(\lambda_{k})
\end{bmatrix}V^\top
$$

Example:
- $f(x) = \frac{1}{x} \rightarrow f(A) = A^{-1}$
- $f(x) = log(x) \rightarrow f(A) = log(A)$

Evaluating spectral functions needs the whole eigenvalue decomposition in general, and is slow.

## Problems, reducing to knowledge about specific spectral functions:
### Heat diffusion in a Network
We can model how heat (or, often information) diffuses in a network over time, using the kernel $f(L) = e^{-tL}$ where we denote the Laplacian of the graph by $L$.

- $y(t) = f(L)x$ describes the temperature of each node, after time $t$, with a starting heat distribution of $x$
- $e_i^\top f(L)e_i $ describes, how much heat node $i$ retained after some time $t$ 

### Gaussian process regression
To compute the log marginal likelyhood during gaussian process regression, one has to compute the log determinant of the covariance matrix $K$.

We know that $log det(K) = Tr(log(K))$.


### Electronic structure calculations
In big electronic structures, like lattices, or crystals, the total energy and possible quantum states are given by a giant Hamiltonian matrix.

Green's function is used in the calculation of the density of states;
For a given energy level $E$, we want to solve for the trace of $G(E)$:
$$
G(E) = (EI - H)^{-1}
$$

Therefore, beeing able to calculate $Tr(A^{-1})$, would clearly be beneficial.



# Quadratic form $x^*f(A)x$
With access to $x^*f(A)x$, we get access to a bunch of things, including individual entries of $f(A)$; $e_i^*f(A)e_j = f(A)_{j,i}$ 

<b>Idea:</b> 
- rewrite $x^*f(A)x$ as an integral over $\mathbb{R}$.
- use estabilished numerical techniques to estimate the value of the integral

# Recall: Gaussian Quadratures
> <b>Gaussian quadrature rule:</b> $\exists w_1 ... w_k$ so that $\int_\mathbb{R}f(x) d\mu(x) \approx \sum_1^k{w_if(\theta_i)}$

Facts:
- the approximation is exact if $f$ is a polynomial of degree at most $2k - 1$.
- the error is controlled by the $2k$th derrivative of $f$

## Legendre polynomials
![legendre_poly.png](legendre_poly.png)
The method relies on the existence of the Legendre polynomials.

Fact:
For each positive measure $\mu$ there exists a system of orthogonal polynomials with respect to the $L_1$ norm.

$$
\exists \{\phi_0, \phi_1, ...\}\\

\int_\mathbb{R}{\phi_i(x)\phi_j(x)d\mu(x)} = 1\cdot\delta_{i,j}
$$

Furthermore, there exist $\gamma_i$, so that:
$$
\gamma_k \phi_k(x) = (x-\xi)\phi_{k-1}(x) - \gamma_{k-1}\phi_{k-2}(x)
$$

Notice, that this looks suspiciously similar to the gram-schmidt orthogonalization done in the Lanczos iteration.

# Using Gaussian Quadratures to estimate $x^*f(A)x$
As suggested previously, we rewrite it as an integral:
$$
x^*f(A)x = \sum_{j=1}^m{f(\lambda_j)x^*P_jx}
$$

We define the measure $\nu$, using $\delta_a$ as the Direc measure at $a$:
$$
\nu = \sum_{j=1}^m{x^*P_jx \cdot \delta_{\lambda_j}}
$$

And get:
$$
x^*f(A)x = \sum_{j=1}^m{f(\lambda_j)x^*P_jx} = \int_\mathbb{R}f(x)d\nu(x)
$$

***

Lets check out why this makes sense;
- The second equality holds trivially; The integral over $\mathbb{R}$, in a sum of dirac measures, is equal to the sum of the dirac measure points.
- We can use gaussian quadratures to estimate the integral: we only need the respective measure to be positive; and $\nu$ is very much positive, as $x^*f(A)x \geq 0$ 

# Lanczos quadrature
We claim, that the Lanczos iteration, is in essence computing orthogonal polynomials. Furthermore, $H_k$ describes the 3-term recurrence, which we already hinted at previously.

This makes sense; We are orthogonalizing $\{\omega, A\omega, ... A^k\omega\}$; Furhtermore, we have seen that $u \in K_k$ implies the existence of a polynomial $\phi$ with degree at most $k$, such that $\phi(A)\omega = u$.

Therefore, there exist $\phi_i$ (called Lanczos polynomials), so that $q_i = \phi_i(A)\omega$. Doing a gram-schmidt orthogonalization of $q_i$ _already feels a lot like computing the legendre polynomials_.

## Calculation
As $AQ = QH$, we get:
$$
Aq_i = H_{(i+1,i)}q_{i+1} + H_{(i,i)}q_i + H_{(i-1, i)}q_{i-1}
$$

We denote $H_{(i,i)}$ by $\alpha_i$, and $H_{(i+1,i)} = H_{(i,i+1)}$ by $b_i$.

which implies:
$$
\beta_i q_{i+1} = (A - I\alpha_i)q_i - \beta_i q_{i-1}
$$

Taking an eigenvector of $A$ as $\omega$ (and remembering that $q_i = \phi_i(A)\omega$), we get $\forall \lambda$:
$$
\beta_i \phi_{i+1}(\lambda) = (\lambda - \alpha_i)\phi_i(\lambda) - \beta_{i-1}\phi_{i-1}(\lambda)
$$

Which is the same 3 term reoccurence we recalled for lagrange polynomials.

## Implementation
Therefore, we can estimate $x^\top f(A)x$:
- We do $k$ steps of the Lanczos iteration on A
- We compute the eigenvalue decomposition of $H_k = U\Theta U^\top$
- We approximate the quadratic form using the Gaussian quadrature rule:
$$
x^\top f(A)x \approx \sum_{i=1}^k{|u_{1i}|^2f(\theta_i)}
$$

# Estimating Tr(f(A))
This leads us to the following realization:

Utilizing the material of the first lecture, 

# References

1.   Joel A. Tropp, *ACM 204: Randomized Algorithms for Matrix Computations*, Caltech, CMS Lecture Notes 2020-01, Pasadena, March 2020.
2.   David I Shuman, Sunil K. Narang, Pascal Frossard, Antonio Ortega, Pierre Vandergheynst, *The Emerging Field of Signal Processing on Graphs*, IEEE Signal Processing Magazine, 30(3), 83–98. 	arXiv:1211.0053
3. 


Matrix size n = 800
Spectral gap (lambda_1 - lambda_2) ≈ 0.001
Target relative error: 1.0e-06

Power iteration (RPM):
  iterations          = 2144
  time                = 0.108 s
  final relative error = 9.98e-07

Randomized Krylov / Lanczos (RKM):
  Lanczos steps       = 23
  time                = 0.003 s
  final relative error = 4.59e-07
